# 🎙️ TranscriptoSense — Colab Pipeline (GPU)
> **FR / AR / Darija** | Audio → Transcription → Semantic Analysis
>
> ⚡ **Make sure Runtime → Change Runtime Type → T4 GPU is selected before running!**

## What this notebook does
1. ✅ Install dependencies
2. ✅ Upload your audio file
3. ✅ Preprocess audio (resample to 16kHz mono)
4. ✅ Voice Activity Detection (VAD)
5. ✅ ASR with Whisper `large-v3` (FR/AR/Darija, word-level timestamps)
6. ✅ Speaker Diarization with pyannote
7. ✅ Merge ASR + Diarization → attributed transcript
8. ✅ Bronze NLP: keywords, decisions, action items (rule-based)
9. ✅ Extractive summary
10. ✅ Export to JSON / text

## 0. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected — go to Runtime → Change Runtime Type → GPU (T4)')

## 1. Install Dependencies

In [ ]:
# This takes ~3-5 minutes on first run
!pip install -q faster-whisper pyannote.audio pydub webrtcvad \
    keybert sentence-transformers langdetect ftfy jiwer rouge-score \
    python-docx reportlab sumy loguru python-dotenv camel-tools

# Download spaCy French model
!python -m spacy download fr_core_news_sm -q
print('✅ Dependencies installed')

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ── Set your HuggingFace token (for pyannote diarization) ──
# Option A: Colab Secrets (recommended — left sidebar → key icon)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
    print('✅ HF token loaded from Colab Secrets')
except Exception:
    HF_TOKEN = ''  # Paste your token here as fallback
    print('⚠️  Add HUGGINGFACE_TOKEN to Colab Secrets for diarization')

# ── Paths ──────────────────────────────────────────────────
WORK_DIR = Path('/content/transcriptosense')
WORK_DIR.mkdir(exist_ok=True)
(WORK_DIR / 'audio').mkdir(exist_ok=True)
(WORK_DIR / 'outputs').mkdir(exist_ok=True)

# ── Model settings ─────────────────────────────────────────
ASR_MODEL    = 'large-v3'  # Best accuracy — needs GPU
DEVICE       = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
COMPUTE_TYPE = 'float16' if DEVICE == 'cuda' else 'int8'

print(f'Device: {DEVICE} | ASR model: {ASR_MODEL} | Compute: {COMPUTE_TYPE}')

## 3. Upload Audio File

In [ ]:
from google.colab import files

print('Upload your audio file (.wav, .mp3, .m4a, .mp4):')
uploaded = files.upload()

AUDIO_FILENAME = list(uploaded.keys())[0]
AUDIO_PATH = WORK_DIR / 'audio' / AUDIO_FILENAME

import shutil
shutil.move(AUDIO_FILENAME, str(AUDIO_PATH))
print(f'✅ Audio saved to: {AUDIO_PATH}')
print(f'   Size: {AUDIO_PATH.stat().st_size / 1e6:.1f} MB')

## 4. Preprocess Audio
> Converts any format to **16kHz mono WAV** (required by Whisper and pyannote)

In [ ]:
from pydub import AudioSegment
import soundfile as sf
import numpy as np

PROCESSED_PATH = WORK_DIR / 'audio' / 'processed_16k.wav'

print(f'Loading: {AUDIO_PATH}')
audio = AudioSegment.from_file(str(AUDIO_PATH))

# Convert to mono 16kHz
audio = audio.set_channels(1).set_frame_rate(16000)
audio.export(str(PROCESSED_PATH), format='wav')

duration_sec = len(audio) / 1000
print(f'✅ Preprocessed: {duration_sec:.1f}s ({duration_sec/60:.1f} min) | 16kHz mono WAV')

## 5. Voice Activity Detection (VAD)
> Identifies which parts of the audio contain speech vs silence/noise

In [ ]:
import webrtcvad
import collections
import contextlib
import struct

def read_wave(path):
    import wave
    with contextlib.closing(wave.open(path, 'rb')) as wf:
        num_channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        sample_rate  = wf.getframerate()
        pcm_data     = wf.readframes(wf.getnframes())
    return pcm_data, sample_rate

def vad_segments(path, aggressiveness=2, frame_duration_ms=30):
    """Returns list of (start_sec, end_sec) speech segments."""
    vad = webrtcvad.Vad(aggressiveness)
    pcm, sample_rate = read_wave(path)
    frame_size = int(sample_rate * frame_duration_ms / 1000) * 2  # 2 bytes/sample

    segments, in_speech, start = [], False, 0
    for i in range(0, len(pcm) - frame_size, frame_size):
        frame = pcm[i:i + frame_size]
        is_speech = vad.is_speech(frame, sample_rate)
        t = i / 2 / sample_rate
        if is_speech and not in_speech:
            start = t
            in_speech = True
        elif not is_speech and in_speech:
            segments.append((round(start, 2), round(t, 2)))
            in_speech = False
    if in_speech:
        segments.append((round(start, 2), round(len(pcm) / 2 / sample_rate, 2)))

    total_speech = sum(e - s for s, e in segments)
    return segments, total_speech

vad_segs, speech_dur = vad_segments(str(PROCESSED_PATH))
print(f'✅ VAD found {len(vad_segs)} speech segments')
print(f'   Total speech: {speech_dur:.1f}s | Silence: {duration_sec - speech_dur:.1f}s')
print(f'   First 5 segments: {vad_segs[:5]}')

## 6. ASR — Whisper large-v3
> Transcribes FR/AR/Darija with word-level timestamps
> ⏱️ ~1-2 min per 10 min of audio on T4 GPU

In [ ]:
from faster_whisper import WhisperModel
import json

print(f'Loading Whisper {ASR_MODEL} on {DEVICE}...')
asr_model = WhisperModel(ASR_MODEL, device=DEVICE, compute_type=COMPUTE_TYPE)
print('✅ Model loaded')

INITIAL_PROMPT = (
    'Transcription en français, arabe et darija marocain. '
    'Les locuteurs peuvent alterner entre les langues (code-switching). '
    'Inclure la ponctuation correcte.'
)

print('Transcribing...')
segments_iter, info = asr_model.transcribe(
    str(PROCESSED_PATH),
    language=None,          # auto-detect
    task='transcribe',
    beam_size=5,
    word_timestamps=True,
    condition_on_previous_text=True,
    initial_prompt=INITIAL_PROMPT,
    vad_filter=True,        # built-in VAD in faster-whisper
)

print(f'  Detected language: {info.language} (probability: {info.language_probability:.2f})')

# Collect all segments
transcript_segments = []
for seg in segments_iter:
    entry = {
        'start':    round(seg.start, 2),
        'end':      round(seg.end, 2),
        'text':     seg.text.strip(),
        'language': info.language,
        'words':    [{'word': w.word, 'start': round(w.start, 2), 'end': round(w.end, 2)}
                     for w in (seg.words or [])]
    }
    transcript_segments.append(entry)
    print(f'  [{entry["start"]:6.1f}s → {entry["end"]:6.1f}s] {entry["text"][:80]}')

# Save raw ASR output
asr_output_path = WORK_DIR / 'outputs' / 'asr_raw.json'
with open(asr_output_path, 'w', encoding='utf-8') as f:
    json.dump({'segments': transcript_segments, 'detected_language': info.language}, f,
              ensure_ascii=False, indent=2)

print(f'\n✅ {len(transcript_segments)} segments transcribed → {asr_output_path}')

## 7. Speaker Diarization — pyannote
> Identifies WHO is speaking at each moment
> ⚠️ Requires HuggingFace token + model acceptance at huggingface.co

In [ ]:
DIARIZATION_AVAILABLE = bool(HF_TOKEN)

if DIARIZATION_AVAILABLE:
    from pyannote.audio import Pipeline
    import torch

    print('Loading pyannote diarization pipeline...')
    diarize_pipeline = Pipeline.from_pretrained(
        'pyannote/speaker-diarization-3.1',
        use_auth_token=HF_TOKEN
    )
    diarize_pipeline = diarize_pipeline.to(torch.device(DEVICE))
    print('✅ Pipeline loaded')

    print('Running diarization...')
    diarization = diarize_pipeline(str(PROCESSED_PATH))

    speaker_segments = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        speaker_segments.append({
            'start':   round(turn.start, 2),
            'end':     round(turn.end, 2),
            'speaker': speaker
        })

    speakers = list(set(s['speaker'] for s in speaker_segments))
    print(f'\n✅ Diarization complete: {len(speakers)} speaker(s) detected — {speakers}')
    print(f'   {len(speaker_segments)} speaker turns')
    for s in speaker_segments[:5]:
        print(f'  [{s["start"]:6.1f}s → {s["end"]:6.1f}s] {s["speaker"]}')
else:
    print('⚠️  Skipping diarization — no HF token. Segments will be unlabeled.')
    speaker_segments = []

## 8. Merge ASR + Diarization
> Assigns a speaker label to each transcription segment

In [ ]:
def assign_speaker(asr_seg, speaker_segs):
    """Assign speaker to an ASR segment by majority overlap."""
    if not speaker_segs:
        return 'UNKNOWN'
    seg_start, seg_end = asr_seg['start'], asr_seg['end']
    overlaps = {}
    for sp in speaker_segs:
        overlap = max(0, min(seg_end, sp['end']) - max(seg_start, sp['start']))
        if overlap > 0:
            overlaps[sp['speaker']] = overlaps.get(sp['speaker'], 0) + overlap
    return max(overlaps, key=overlaps.get) if overlaps else 'UNKNOWN'

attributed = []
for seg in transcript_segments:
    seg_copy = dict(seg)
    seg_copy['speaker'] = assign_speaker(seg, speaker_segments)
    attributed.append(seg_copy)

# Save attributed transcript
attr_path = WORK_DIR / 'outputs' / 'transcript_attributed.json'
with open(attr_path, 'w', encoding='utf-8') as f:
    json.dump({'segments': attributed}, f, ensure_ascii=False, indent=2)

print('\n✅ Speaker-attributed transcript:')
for s in attributed[:8]:
    ts = f"[{s['start']:.1f}s]"
    print(f'  {ts:10} [{s["speaker"]:12}] {s["text"][:70]}')
print(f'\nSaved → {attr_path}')

## 9. Bronze NLP — Keywords, Decisions & Action Items
> Rule-based extraction (no GPU needed, runs fast)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Rule patterns ────────────────────────────────────────────
DECISION_PATTERNS = [
    r"\bon a décidé\b", r"\bil est décidé\b", r"\bnous allons\b",
    r"\bla décision\b", r"\bconclusion\b", r"\bvalidé\b", r"\baccord sur\b",
    r"\bتم الاتفاق\b", r"\bقررنا\b", r"\bتقرر\b",
]
ACTION_PATTERNS = [
    r"\bil faut\b", r"\btu dois\b", r"\bvous devez\b", r"\bà faire\b",
    r"\baction\b", r"\btâche\b", r"\bresponsable\b", r"\bje vais\b",
    r"\bon va\b", r"\bprendre en charge\b", r"\bdelai\b",
    r"\byجب\b", r"\bلازم\b", r"\bkhassna\b", r"\bghadi ndiru\b",
]

def detect_category(text):
    t = text.lower()
    if any(re.search(p, t) for p in DECISION_PATTERNS):
        return 'decision'
    if any(re.search(p, t) for p in ACTION_PATTERNS):
        return 'action_item'
    return None

# ── TF-IDF Keywords ──────────────────────────────────────────
all_texts = [s['text'] for s in attributed if len(s['text']) > 10]
keywords = []
if all_texts:
    tfidf = TfidfVectorizer(max_features=20, stop_words=None, ngram_range=(1, 2))
    try:
        tfidf.fit(all_texts)
        keywords = tfidf.get_feature_names_out().tolist()
    except Exception as e:
        keywords = []
        print(f'⚠️  TF-IDF failed: {e}')

# ── Annotate each segment ────────────────────────────────────
decisions    = []
action_items = []

for seg in attributed:
    category = detect_category(seg['text'])
    seg['category'] = category
    if category == 'decision':
        decisions.append({'text': seg['text'], 'speaker': seg['speaker'],
                          'timestamp': seg['start']})
    elif category == 'action_item':
        action_items.append({'text': seg['text'], 'speaker': seg['speaker'],
                             'timestamp': seg['start']})

print(f'✅ NLP extraction complete')
print(f'   Keywords ({len(keywords)}): {keywords[:10]}')
print(f'   Decisions found: {len(decisions)}')
print(f'   Action items found: {len(action_items)}')

for d in decisions[:3]:
    print(f'  📌 [{d["timestamp"]:.1f}s] {d["speaker"]}: {d["text"][:80]}')
for a in action_items[:3]:
    print(f'  ✅ [{a["timestamp"]:.1f}s] {a["speaker"]}: {a["text"][:80]}')

## 10. Extractive Summary (Bronze)
> Picks the most important sentences from the transcript

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer

full_text = ' '.join(s['text'] for s in attributed if s['text'].strip())

try:
    parser    = PlaintextParser.from_string(full_text, Tokenizer('french'))
    stemmer   = Stemmer('french')
    summarizer = LsaSummarizer(stemmer)
    summary_sentences = summarizer(parser.document, sentences_count=5)
    summary = ' '.join(str(s) for s in summary_sentences)
except Exception as e:
    # Fallback: first 5 longest sentences
    sentences = [s['text'] for s in attributed if len(s['text']) > 30]
    sentences.sort(key=len, reverse=True)
    summary = ' '.join(sentences[:5])
    print(f'⚠️  sumy failed ({e}), using fallback summary')

print('\n📝 Extractive Summary:')
print('─' * 60)
print(summary)
print('─' * 60)

## 11. Export — Meeting Minutes JSON

In [ ]:
import datetime

meeting_minutes = {
    'meta': {
        'generated_at': datetime.datetime.now().isoformat(),
        'audio_file':   AUDIO_FILENAME,
        'duration_sec': duration_sec,
        'detected_language': info.language,
        'pipeline': 'TranscriptoSense Bronze v0.1'
    },
    'summary': summary,
    'keywords': keywords,
    'decisions': decisions,
    'action_items': action_items,
    'transcript': attributed
}

minutes_path = WORK_DIR / 'outputs' / 'meeting_minutes.json'
with open(minutes_path, 'w', encoding='utf-8') as f:
    json.dump(meeting_minutes, f, ensure_ascii=False, indent=2)

print(f'✅ Meeting minutes saved → {minutes_path}')
print(f'\nContents:')
print(f'  Transcript segments : {len(attributed)}')
print(f'  Decisions           : {len(decisions)}')
print(f'  Action items        : {len(action_items)}')
print(f'  Keywords            : {len(keywords)}')

## 12. Download Results

In [ ]:
import shutil
from google.colab import files

# Zip all outputs
zip_path = '/content/transcriptosense_outputs'
shutil.make_archive(zip_path, 'zip', str(WORK_DIR / 'outputs'))
files.download(zip_path + '.zip')
print('✅ Downloaded transcriptosense_outputs.zip')

---
## ✅ Next Steps

| Tier | What to add |
|---|---|
| 🥈 Silver | NER with `Davlan/bert-base-multilingual-cased-ner-hrl`, topic segmentation with embeddings, query-focused summarization with mT5 |
| 🥇 Gold | Fine-tune action item classifier, SRL/SVO extraction, ASR calibration (confidence scores) |
| 💎 Platinum | Streaming pipeline, drift detection, LoRA fine-tuning on Darija data |

📁 Move your `meeting_minutes.json` to `outputs/transcripts/` in your local project folder.